# Codebase AI pipeline demo

This notebook calls the Python functions in ai-service directly. It does not start or call the FastAPI app. It loads a public GitHub repository, previews the files and chunks, writes and reads vectors in an in-memory Chroma collection, then optionally asks Gemini a question.

Use a Python environment with the packages from ai-service/requirements.txt installed. The Gemini step needs GOOGLE_API_KEY in the environment or in the project .env / ai-service/.env file. No ChromaDB server is needed for this notebook.


In [1]:
import os
import sys
from html import escape
from pathlib import Path

# Find the project root even if Jupyter was started from a subfolder.
PROJECT_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / "ai-service" / "repo_loader.py").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find ai-service/repo_loader.py from the current directory.")

AI_SERVICE_DIR = PROJECT_ROOT / "ai-service"
if str(AI_SERVICE_DIR) not in sys.path:
    sys.path.insert(0, str(AI_SERVICE_DIR))

# Load local keys without printing them. Existing environment variables take precedence.
from dotenv import load_dotenv
load_dotenv(AI_SERVICE_DIR / ".env")
load_dotenv(PROJECT_ROOT / ".env")

# Render tables in Jupyter; keep a plain-text fallback for running cells in a Python console.
try:
    from IPython.display import HTML, display
    IN_NOTEBOOK = True
except ImportError:
    IN_NOTEBOOK = False
    def display(value):
        print(value)
if not IN_NOTEBOOK and hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")

def show_table(rows):
    rows = list(rows)
    if not rows:
        print("No rows to display.")
        return
    columns = list(rows[0])
    if IN_NOTEBOOK:
        header = "".join(f"<th>{escape(str(column))}</th>" for column in columns)
        body = "".join(
            "<tr>" + "".join(
                f'<td style="vertical-align:top; padding:6px; border:1px solid #ddd;">{escape(str(row.get(column, "")))}</td>'
                for column in columns
            ) + "</tr>"
            for row in rows
        )
        display(HTML(
            '<div style="overflow-x:auto"><table style="border-collapse:collapse; width:100%;">'
            f'<thead><tr>{header}</tr></thead><tbody>{body}</tbody></table></div>'
        ))
    else:
        print(" | ".join(columns))
        for row in rows:
            print(" | ".join(str(row.get(column, "")) for column in columns))

print(f"Project root: {PROJECT_ROOT}")
print("Notebook mode: direct Python function calls (FastAPI is not used).")


Project root: D:\Projects\Codebase Chat Assistant
Notebook mode: direct Python function calls (FastAPI is not used).


In [2]:
# Change the GitHub URL to inspect another public repository.
GITHUB_REPO_URL = "https://github.com/omkarbg100/codebase-chat-assistant"
QUESTION = "What are the main components of this codebase and how does the AI retrieval flow work?"

import repo_loader
import chunker

print(f"Repository selected: {GITHUB_REPO_URL}")
print("Imported repo_loader and chunker directly from ai-service.")


Repository selected: https://github.com/omkarbg100/codebase-chat-assistant
Imported repo_loader and chunker directly from ai-service.


## 1. Load and preview repository files

clone_repo downloads the public GitHub archive. get_repo_files applies the service's file and directory filters. The temporary checkout is removed after the file contents are read.


In [3]:
import time
TEMP_REPO_DIR, REPO_ID = repo_loader.clone_repo(GITHUB_REPO_URL)
try:
    REPO_FILES = list(repo_loader.get_repo_files(TEMP_REPO_DIR))
finally:
    TEMP_REPO_CLEANED = False
    for cleanup_attempt in range(20):
        try:
            repo_loader.cleanup_repo(TEMP_REPO_DIR)
            TEMP_REPO_CLEANED = True
            break
        except PermissionError as cleanup_error:
            if cleanup_attempt == 19:
                print(f"Temporary checkout cleanup deferred: {cleanup_error}")
            else:
                time.sleep(0.5)

assert REPO_FILES, "No supported source files were loaded from the repository."
TOTAL_SOURCE_CHARS = sum(len(item["content"]) for item in REPO_FILES)
print(f"repo_id: {REPO_ID}")
print(f"Source files loaded: {len(REPO_FILES)}")
print(f"Total source characters: {TOTAL_SOURCE_CHARS:,}")
print("Temporary checkout cleaned up." if TEMP_REPO_CLEANED else "Temporary checkout cleanup can be retried with repo_loader.cleanup_repo(TEMP_REPO_DIR).")
nested_temp_files = [
    item["metadata"].get("file_path", "")
    for item in REPO_FILES
    if "/temp_repos/" in item["metadata"].get("file_path", "").replace("\\", "/")
]
print(f"Files found under nested temp_repos directories: {len(nested_temp_files)}")
if nested_temp_files:
    show_table([{"nested_temp_repo_file": file_path} for file_path in nested_temp_files[:5]])

file_preview = [
    {
        "file_path": item["metadata"].get("file_path", ""),
        "language": item["metadata"].get("language", ""),
        "characters": len(item["content"]),
        "preview": item["content"][:180].replace("\n", " "),
    }
    for item in REPO_FILES[:15]
]
show_table(file_preview)


repo_id: aee230c9-9b95-4d11-bded-ed68a939cd66
Source files loaded: 36
Total source characters: 80,687
Temporary checkout cleaned up.
Files found under nested temp_repos directories: 0
file_path | language | characters | preview
omkarbg100-Codebase-Chat-Assistant-0d0cf40\cicd_setup_guide.md | md | 3447 | # CI/CD Setup Guide — Codebase Chat Assistant  ## What Was Created  | File | Purpose | |---|---| | [.github/workflows/deploy.yml](file:///d:/Projects/Codebase%20Chat%20Assistant/.g
omkarbg100-Codebase-Chat-Assistant-0d0cf40\README.md | md | 10210 | <div align="center">  # 🧑‍💻 Codebase Chat Assistant  **Chat with any GitHub repository using AI.**   Upload a repo, ask questions in plain English, and get context-aware answers wi
omkarbg100-Codebase-Chat-Assistant-0d0cf40\ai-service\chunker.py | py | 928 | from langchain_text_splitters import RecursiveCharacterTextSplitter  def chunk_code(files_data):     """Splits code files into chunks while preserving metadata."""     # Using a mi
omkarb

## 2. Chunk the source code

This exercises chunk_code with its configured chunk size, overlap, separators, and per-file metadata. The summary and first chunk previews make the transformation visible.


In [4]:
from collections import Counter

CHUNKS = chunker.chunk_code(REPO_FILES)
assert CHUNKS, "The chunker returned no chunks."
assert all(chunk.get("page_content") and chunk.get("metadata", {}).get("file_path") for chunk in CHUNKS)

chunks_per_file = Counter(chunk["metadata"]["file_path"] for chunk in CHUNKS)
print(f"Chunks created: {len(CHUNKS)}")
print(f"Files represented in chunks: {len(chunks_per_file)}")
print(f"Longest chunk: {max(len(chunk['page_content']) for chunk in CHUNKS):,} characters")
OVERSIZED_CHUNKS = [chunk for chunk in CHUNKS if len(chunk["page_content"]) > 1500]
print(f"Chunks exceeding the configured 1,500-character target: {len(OVERSIZED_CHUNKS)}")
if OVERSIZED_CHUNKS:
    show_table([
        {
            "file_path": chunk["metadata"].get("file_path", ""),
            "chunk_id": chunk["metadata"].get("chunk_id", ""),
            "characters": len(chunk["page_content"]),
        }
        for chunk in sorted(OVERSIZED_CHUNKS, key=lambda item: len(item["page_content"]), reverse=True)[:10]
    ])

chunk_preview = [
    {
        "file_path": chunk["metadata"].get("file_path", ""),
        "chunk_id": chunk["metadata"].get("chunk_id", ""),
        "characters": len(chunk["page_content"]),
        "preview": chunk["page_content"][:220].replace("\n", " "),
    }
    for chunk in CHUNKS[:15]
]
show_table(chunk_preview)

print("Most chunked files:")
show_table([{"file_path": path, "chunk_count": count} for path, count in chunks_per_file.most_common(10)])


Chunks created: 94
Files represented in chunks: 36
Longest chunk: 1,498 characters
Chunks exceeding the configured 1,500-character target: 0
file_path | chunk_id | characters | preview
omkarbg100-Codebase-Chat-Assistant-0d0cf40\cicd_setup_guide.md | 0 | 1352 | # CI/CD Setup Guide — Codebase Chat Assistant  ## What Was Created  | File | Purpose | |---|---| | [.github/workflows/deploy.yml](file:///d:/Projects/Codebase%20Chat%20Assistant/.github/workflows/deploy.yml) | GitHub Act
omkarbg100-Codebase-Chat-Assistant-0d0cf40\cicd_setup_guide.md | 1 | 1311 | ---  ## Required GitHub Secrets  Go to **GitHub → Your Repo → Settings → Secrets and variables → Actions → New repository secret** and add:  | Secret Name | Value | |---|---| | `DOCKERHUB_USERNAME` | Your Docker Hub user
omkarbg100-Codebase-Chat-Assistant-0d0cf40\cicd_setup_guide.md | 2 | 1079 | # 4. Create the app directory mkdir -p ~/codebase-chat-assistant ```  ---  ## Docker Hub Image Names  After the first successful run, your images

## 3. Create and reload a local vector store

create_vector_store and get_vector_store are called directly. The service module normally uses a remote ChromaDB server; this notebook replaces only its client factory with Chroma's in-memory client so no service or container has to be started. The embedding model still comes from vector_store.py.


In [5]:
import chromadb
import vector_store as vector_store_module

NOTEBOOK_CHROMA_CLIENT = chromadb.EphemeralClient()
vector_store_module.get_chroma_client = lambda: NOTEBOOK_CHROMA_CLIENT

CREATED_VECTOR_STORE = vector_store_module.create_vector_store(CHUNKS, REPO_ID)
LOADED_VECTOR_STORE = vector_store_module.get_vector_store(REPO_ID)
assert LOADED_VECTOR_STORE is not None, "Could not reload the in-memory vector collection."
stored_ids = LOADED_VECTOR_STORE.get().get("ids", [])
assert len(stored_ids) == len(CHUNKS), f"Expected {len(CHUNKS)} stored chunks, found {len(stored_ids)}."

print("Vector store create/reload check passed.")
print(f"Stored vectors: {len(stored_ids)}")
print(f"Collection: {REPO_ID}")


🚀 Creating vector store collection for repo: aee230c9-9b95-4d11-bded-ed68a939cd66
📊 Total chunks: 94
⚡ Processing batch 1
✅ Vector store created successfully!
📂 Loading existing vector DB collection for repo: aee230c9-9b95-4d11-bded-ed68a939cd66
Vector store create/reload check passed.
Stored vectors: 94
Collection: aee230c9-9b95-4d11-bded-ed68a939cd66


## 4. Retrieve relevant chunks and ask a question

The retrieval preview runs even without a Google key. If GOOGLE_API_KEY is available, the last cell also calls get_rag_chain directly and prints its Gemini answer. Follow-up questions can be asked by calling ask_repo(...) again; this notebook keeps the LangChain message history in memory.


In [6]:
from langchain_core.messages import AIMessage, HumanMessage

retriever = LOADED_VECTOR_STORE.as_retriever(search_kwargs={"k": 5})
RETRIEVED_DOCS = retriever.invoke(QUESTION)
assert RETRIEVED_DOCS, "Vector retrieval returned no chunks."
print(f"Retrieved {len(RETRIEVED_DOCS)} chunks for the question:")
print(QUESTION)
show_table([
    {
        "rank": index,
        "file_path": doc.metadata.get("file_path", ""),
        "language": doc.metadata.get("language", ""),
        "preview": doc.page_content[:260].replace("\n", " "),
    }
    for index, doc in enumerate(RETRIEVED_DOCS, start=1)
])

CHAT_HISTORY = []
RAG_CHAIN = None

def ask_repo(question):
    """Run the imported RAG chain directly and retain the conversation in this kernel."""
    global RAG_CHAIN
    if not os.getenv("GOOGLE_API_KEY") or os.getenv("GOOGLE_API_KEY") == "your_gemini_api_key_here":
        raise RuntimeError("Set GOOGLE_API_KEY in the environment or a local .env file before asking Gemini.")
    if RAG_CHAIN is None:
        import rag_pipeline
        RAG_CHAIN = rag_pipeline.get_rag_chain(LOADED_VECTOR_STORE)
    answer = RAG_CHAIN.invoke({"input": question, "chat_history": CHAT_HISTORY})
    CHAT_HISTORY.extend([HumanMessage(content=question), AIMessage(content=answer)])
    return answer

if os.getenv("GOOGLE_API_KEY") and os.getenv("GOOGLE_API_KEY") != "your_gemini_api_key_here":
    ANSWER = ask_repo(QUESTION)
    print("\nGemini answer:\n")
    print(ANSWER)
else:
    ANSWER = None
    print("Gemini call skipped: GOOGLE_API_KEY is not configured. Retrieval was tested above.")

print("Direct checks completed: repository loading, file reading, chunking, vector create/reload, and retrieval.")


Retrieved 5 chunks for the question:
What are the main components of this codebase and how does the AI retrieval flow work?
rank | file_path | language | preview
1 | omkarbg100-Codebase-Chat-Assistant-0d0cf40\README.md | md | ---  ## 🏗️ Architecture  ``` ┌─────────────────────────────────────────────────────────┐ │                        Browser                          │ │              http://localhost:3000                      │ └────────────────────────┬─────────────────────────
2 | omkarbg100-Codebase-Chat-Assistant-0d0cf40\ai-service\main.py | py | # pyrefly: ignore [missing-import] from fastapi import FastAPI, HTTPException # pyrefly: ignore [missing-import] from pydantic import BaseModel import os # pyrefly: ignore [missing-import] import uvicorn from repo_loader import clone_repo, get_repo_files, clea
3 | omkarbg100-Codebase-Chat-Assistant-0d0cf40\README.md | md | <div align="center">  # 🧑‍💻 Codebase Chat Assistant  **Chat with any GitHub repository using AI.**   Upload a repo,